# SDPO 训练（BF16，无 GradScaler 版）

避免 `_amp_foreach_non_finite_check_and_unscale_cuda` 报错，使用 bfloat16 autocast 并取消 GradScaler。

In [ ]:
import os
os.chdir('/data/coding/ARC')
import torch
from itertools import cycle
from torch.cuda.amp import autocast

from ARChitects.architect_sdpo import SDPOConfig, load_model_and_tokenizer, load_sdpo_dataset, build_dataloader, SDPOTrainer
from eval.core_sdpo import sdpo_forward

# 配置可按需修改
config = SDPOConfig(
    model_path='outputs/arc_lora_sft_C/lora_merged_model',
    tokenizer_path='outputs/arc_lora_sft_C/lora_merged_model',
    report_path='outputs/architect_eval/report_architect_trans_candall.json',
    dataset_root='data/evaluation',
    batch_size=2,
    learning_rate=5e-6,
    max_steps=100,
)
device = torch.device(config.device)
print(config)

model, tokenizer = load_model_and_tokenizer(config)
model.config.use_cache = False  # 训练关闭 cache 节省显存
dataset = load_sdpo_dataset(config)
print(f"trainable examples: {len(dataset)}")
dataloader = build_dataloader(tokenizer, dataset, device=device, batch_size=config.batch_size)
trainer = SDPOTrainer(model, tokenizer, config)

In [ ]:
# BF16 训练循环（无 GradScaler）
def train_loop_bf16(
    trainer: SDPOTrainer,
    train_loader,
    *,
    valid_loader=None,
    max_steps=None,
    log_every=10,
    eval_every=100,
):
    max_steps = max_steps or trainer.config.max_steps
    step = 0
    train_iter = cycle(train_loader)
    opt = trainer.optimizer
    sch = trainer.lr_scheduler

    while step < max_steps:
        batch = next(train_iter)
        opt.zero_grad(set_to_none=True)
        with autocast(dtype=torch.bfloat16):
            loss, _ = sdpo_forward(
                trainer.model,
                trainer.tokenizer,
                batch,
                loss_type=trainer.config.loss_type,
                top_k=trainer.config.top_k,
            )
        loss.backward()
        if trainer.config.grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(trainer.model.parameters(), trainer.config.grad_clip)
        opt.step()
        sch.step()
        step += 1

        if step % log_every == 0:
            print(f"[sdpo] step {step}/{max_steps} train_loss={loss.item():.4f}")

        if valid_loader is not None and step % eval_every == 0:
            try:
                vbatch = next(iter(valid_loader))
            except StopIteration:
                vbatch = next(iter(valid_loader))
            with torch.no_grad(), autocast(dtype=torch.bfloat16):
                v_loss, _ = sdpo_forward(
                    trainer.model,
                    trainer.tokenizer,
                    vbatch,
                    loss_type=trainer.config.loss_type,
                    top_k=trainer.config.top_k,
                )
            print(f"[sdpo] eval step {step}: loss={v_loss.item():.4f}")

    print(f"[sdpo] training done: steps={step}")
    return trainer


In [ ]:
# 运行若干步示例
_ = train_loop_bf16(
    trainer,
    dataloader,
    valid_loader=None,
    max_steps=50,
    log_every=5,
    eval_every=25,
)
